In [1]:
import h5py

#查看数据集内的结构

def explore_h5(file_path):
    """递归打印 HDF5 文件结构"""
    def print_structure(name, obj):
        indent = '  ' * (name.count('/'))
        if isinstance(obj, h5py.Dataset):
            print(f"{indent}├── 数据集: {name.split('/')[-1]} (形状: {obj.shape}, 数据类型: {obj.dtype})")
        elif isinstance(obj, h5py.Group):
            print(f"{indent}└── 组: {name.split('/')[-1]}")

    with h5py.File(file_path, 'r') as f:
        print(f"文件: {file_path}")
        f.visititems(print_structure)
        print("\n详细数据集信息:")
        f.visititems(lambda name, obj: print(f"  {name}: {obj.shape} | {obj.dtype}") if isinstance(obj, h5py.Dataset) else None)

# 使用示例
explore_h5('CWRU_base.h5')

文件: CWRU_base.h5
├── 数据集: X_test (形状: (2200, 1, 1024, 1), 数据类型: float64)
├── 数据集: X_train (形状: (2200, 1, 1024, 1), 数据类型: float64)
├── 数据集: X_val (形状: (2200, 1, 1024, 1), 数据类型: float64)
├── 数据集: y_test (形状: (2200,), 数据类型: uint8)
├── 数据集: y_train (形状: (2200,), 数据类型: uint8)
├── 数据集: y_val (形状: (2200,), 数据类型: uint8)

详细数据集信息:
  X_test: (2200, 1, 1024, 1) | float64
  X_train: (2200, 1, 1024, 1) | float64
  X_val: (2200, 1, 1024, 1) | float64
  y_test: (2200,) | uint8
  y_train: (2200,) | uint8
  y_val: (2200,) | uint8


In [2]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

# 设置参数
SNR_DB = 0  # 目标信噪比(dB)，可根据需要调整
SEED = 42  # 随机种子确保可重复性
np.random.seed(SEED)

In [3]:
# 1. 读取原始数据
print("正在读取原始数据...")
with h5py.File('CWRU_base.h5', 'r') as f:
    # 读取原始数据
    X_train = f['X_train'][:]
    X_val = f['X_val'][:]
    X_test = f['X_test'][:]
    y_train = f['y_train'][:]
    y_val = f['y_val'][:]
    y_test = f['y_test'][:]
    
print(f"数据读取完成: 训练集 {X_train.shape}, 验证集 {X_val.shape}, 测试集 {X_test.shape}")

正在读取原始数据...
数据读取完成: 训练集 (2200, 1, 1024, 1), 验证集 (2200, 1, 1024, 1), 测试集 (2200, 1, 1024, 1)


In [4]:
# 2. 定义自适应噪声添加函数
def add_adaptive_noise(data, snr_db=SNR_DB):
    """
    添加自适应噪声，根据信噪比(SNR)动态调整噪声强度
    snr_db: 目标信噪比(dB)
    """
    # 创建数据副本以避免修改原始数据
    noisy_data = np.copy(data)
    
    # 获取数据形状信息
    n_samples = data.shape[0]
    signal_length = data.shape[2]
    
    # 添加进度条
    for i in tqdm(range(n_samples), desc="添加自适应噪声"):
        # 提取当前样本信号 (形状: [1, 1024, 1])
        signal = data[i, 0, :, 0]
        
        # 计算信号功率 (均方值)
        signal_power = np.mean(signal ** 2)
        
        # 跳过零功率信号
        if signal_power < 1e-10:
            noisy_data[i] = data[i]
            continue
            
        # 计算目标噪声功率
        snr_linear = 10 ** (snr_db / 10)
        noise_power = signal_power / snr_linear
        
        # 生成高斯噪声
        noise = np.random.normal(0, np.sqrt(noise_power), signal_length)
        
        # 添加噪声到信号
        noisy_signal = signal + noise
        
        # 更新数据
        noisy_data[i, 0, :, 0] = noisy_signal
    
    return noisy_data

In [5]:
# 3. 创建噪声训练集和测试集
print("\n创建噪声训练集...")
X_train_noisy = add_adaptive_noise(X_train, snr_db=SNR_DB)

print("\n创建噪声测试集...")
X_test_noisy = add_adaptive_noise(X_test, snr_db=SNR_DB)

# 4. 合并原始训练集和噪声训练集
print("\n合并训练集...")
X_train_combined = np.concatenate((X_train, X_train_noisy), axis=0)
y_train_combined = np.concatenate((y_train, y_train), axis=0)

# 打乱合并后的数据集
print("\n打乱合并数据集...")
indices = np.arange(X_train_combined.shape[0])
np.random.shuffle(indices)
X_train_combined = X_train_combined[indices]
y_train_combined = y_train_combined[indices]


创建噪声训练集...


添加自适应噪声: 100%|███████████████████████████████████████████████████████████| 2200/2200 [00:00<00:00, 21222.57it/s]



创建噪声测试集...


添加自适应噪声: 100%|███████████████████████████████████████████████████████████| 2200/2200 [00:00<00:00, 20253.27it/s]


合并训练集...

打乱合并数据集...


In [6]:
# 5. 保存所有场景到HDF5文件
def save_h5_dataset(filename, X_train, X_val, X_test, y_train, y_val, y_test):
    """保存数据集到HDF5文件"""
    with h5py.File(filename, 'w') as f:
        f.create_dataset('X_train', data=X_train)
        f.create_dataset('X_val', data=X_val)
        f.create_dataset('X_test', data=X_test)
        f.create_dataset('y_train', data=y_train)
        f.create_dataset('y_val', data=y_val)
        f.create_dataset('y_test', data=y_test)
    print(f"已保存: {filename}")

print("\n保存数据集...")
# 场景1: 只在训练集加噪声
#save_h5_dataset('CWRU_train_noisy_SNR20.h5', 
                #X_train_noisy, X_val, X_test,
                #y_train, y_val, y_test)

# 场景2: 只在测试集加噪声
save_h5_dataset('CWRU_test_noisy_SNR20.h5', 
                X_train, X_val, X_test_noisy,
                y_train, y_val, y_test)

# 场景3: 训练集和测试集都加噪声
#save_h5_dataset('CWRU_train_test_noisy_SNR20.h5', 
                #X_train_noisy, X_val, X_test_noisy,
                #y_train, y_val, y_test)

# 场景4: 合并训练集，测试集不添加噪声
#save_h5_dataset('Double_train_noisy_SNR20.h5', 
                #X_train_combined, X_val, X_test,
                #y_train_combined, y_val, y_test)

# 场景5: 合并训练集，测试集添加噪声
#save_h5_dataset('Double_train_test_noisy_SNR20.h5', 
                #X_train_combined, X_val, X_test_noisy,
                #y_train_combined, y_val, y_test)



保存数据集...
已保存: CWRU_test_noisy_SNR20.h5


In [7]:
# 6. 可视化函数 - 用于单个样本对比
def plot_single_sample_comparison(original, noisy, title, filename):
    """绘制单个样本的原始信号和噪声信号对比"""
    plt.figure(figsize=(12, 8))
    
    # 原始信号
    plt.subplot(3, 1, 1)
    plt.plot(original)
    plt.title(f"Original Signal")  # 原始信号
    plt.xlabel("Time Points")  # 时间点
    plt.ylabel("Amplitude")  # 振幅
    
    # 噪声信号
    plt.subplot(3, 1, 2)
    plt.plot(noisy)
    plt.title(f"Noisy Signal (SNR={SNR_DB}dB)")  # 添加噪声后的信号
    plt.xlabel("Time Points")  # 时间点
    plt.ylabel("Amplitude")  # 振幅
    
    # 噪声分量
    plt.subplot(3, 1, 3)
    noise_component = noisy - original
    plt.plot(noise_component)
    plt.title(f"Noise Component")  # 噪声分量
    plt.xlabel("Time Points")  # 时间点
    plt.ylabel("Amplitude")  # 振幅
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()  # 关闭图形以节省内存

In [8]:
# 7. 场景可视化对比
print("\n开始对每个场景进行可视化对比...")
sample_idx = np.random.randint(0, 2200)  # 随机选择一个样本

# 场景1: 只在训练集加噪声
#print("可视化场景1: 只在训练集加噪声")
#plot_single_sample_comparison(
    #X_train[sample_idx, 0, :, 0].flatten(),
    #X_train_noisy[sample_idx, 0, :, 0].flatten(),
    #"训练集样本 (场景1)",
    #'train_noise_SNR=20.png'
#)

# 场景2: 只在测试集加噪声
print("可视化场景2: 只在测试集加噪声")
plot_single_sample_comparison(
    X_test[sample_idx, 0, :, 0].flatten(),
    X_test_noisy[sample_idx, 0, :, 0].flatten(),
    "测试集样本 (场景2)",
    'test_noise_SNR=20.png'
)


开始对每个场景进行可视化对比...
可视化场景2: 只在测试集加噪声


In [9]:
# 8. 创建综合对比图
print("\n创建综合对比图...")
def create_comprehensive_comparison():
    """创建所有场景的综合对比图"""
    plt.figure(figsize=(15, 20))
    
    # 原始训练集样本
    plt.subplot(5, 3, 1)
    plt.plot(X_train[sample_idx, 0, :, 0].flatten())
    plt.title("Original training set samples")
    plt.xlabel("Time Points")
    plt.ylabel("Amplitude")
    
    # 场景1: 训练集添加噪声
    #plt.subplot(5, 3, 2)
    #plt.plot(X_train_noisy[sample_idx, 0, :, 0].flatten())
    #plt.title("scene1: Training set noise samples")
    #plt.xlabel("Time Points")
    
    #plt.subplot(5, 3, 3)
    #noise = X_train_noisy[sample_idx, 0, :, 0].flatten() - X_train[sample_idx, 0, :, 0].flatten()
    #plt.plot(noise)
    #plt.title("scene1:Noise Component")
    #plt.xlabel("Time Points")
    
    # 场景2: 测试集添加噪声
    plt.subplot(5, 3, 4)
    plt.plot(X_test[sample_idx, 0, :, 0].flatten())
    plt.title("Original testing set samples")
    plt.ylabel("Amplitude")
    
    plt.subplot(5, 3, 5)
    plt.plot(X_test_noisy[sample_idx, 0, :, 0].flatten())
    plt.title("scene2: Training set noise samples")
    
    plt.subplot(5, 3, 6)
    noise = X_test_noisy[sample_idx, 0, :, 0].flatten() - X_test[sample_idx, 0, :, 0].flatten()
    plt.plot(noise)
    plt.title("scene2: Noise Component")
       
    plt.tight_layout()
    plt.savefig('comparison_SNR=20.png', dpi=300)
    plt.close()

create_comprehensive_comparison()


创建综合对比图...


In [10]:
# 9. 验证文件内容
def verify_h5_file(filename):
    """验证HDF5文件内容"""
    print(f"\n验证文件: {filename}")
    with h5py.File(filename, 'r') as f:
        print(f"X_train shape: {f['X_train'].shape}")
        print(f"X_test shape: {f['X_test'].shape}")
        print(f"y_train shape: {f['y_train'].shape}")
        print(f"y_test shape: {f['y_test'].shape}")
        
        # 计算信噪比
        sample_idx = np.random.randint(0, 2200)
        orig_signal = f['X_test'][sample_idx, 0, :, 0]
        
        # 如果是噪声文件，计算实际SNR
        if "noisy" in filename:
            # 获取原始数据中的对应信号
            with h5py.File('CWRU_base.h5', 'r') as orig_f:
                # 对于合并数据集，需要处理索引
                if "Double" in filename:
                    # 在合并数据集中，前2200个是原始样本，后2200个是噪声样本
                    if sample_idx < 2200:
                        orig_signal_clean = orig_f['X_train'][sample_idx, 0, :, 0]
                    else:
                        orig_signal_clean = orig_f['X_train'][sample_idx-2200, 0, :, 0]
                else:
                    orig_signal_clean = orig_f['X_train'][sample_idx, 0, :, 0]
            
            noise = orig_signal - orig_signal_clean
            signal_power = np.mean(orig_signal_clean ** 2)
            noise_power = np.mean(noise ** 2)
            
            if noise_power > 1e-10:
                snr = 10 * np.log10(signal_power / noise_power)
                print(f"样本 {sample_idx} 实际信噪比: {snr:.2f} dB")
            else:
                print("噪声功率过低，无法计算SNR")

print("\n验证生成的文件...")
#verify_h5_file('CWRU_train_noisy_SNR20.h5')
verify_h5_file('CWRU_test_noisy_SNR20.h5')
#verify_h5_file('CWRU_train_test_noisy_SNR20.h5')
#verify_h5_file('Double_train_noisy_SNR20.h5')
#verify_h5_file('Double_train_test_noisy_SNR20.h5')


验证生成的文件...

验证文件: CWRU_test_noisy_SNR20.h5
X_train shape: (2200, 1, 1024, 1)
X_test shape: (2200, 1, 1024, 1)
y_train shape: (2200,)
y_test shape: (2200,)
样本 2115 实际信噪比: -3.31 dB


In [11]:
# 10. 打印合并数据集信息
print("\n合并数据集统计:")
print(f"原始训练集大小: {X_train.shape[0]} 样本")
print(f"噪声训练集大小: {X_train_noisy.shape[0]} 样本")
print(f"合并后训练集大小: {X_train_combined.shape[0]} 样本")

print("\n所有操作完成！已保存所有场景的可视化对比图。")


合并数据集统计:
原始训练集大小: 2200 样本
噪声训练集大小: 2200 样本
合并后训练集大小: 4400 样本

所有操作完成！已保存所有场景的可视化对比图。
